# Module 5 — Campaign incrementality

This follows Word guide Steps 1.5.1–1.5.5. The campaign assignment was observational, not randomized. We compare four designs, use fake-start placebo checks to look for bias, and collect inputs for the later ROI model. Dollar estimates are per household per week.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
OUT = ROOT / "exports" / "campaigns"
def load(name):
    return pd.read_csv(OUT / f"{name}.csv")
overview = load("campaign_overview")
display(overview[["campaign_id", "campaign_type", "start_date", "end_date", "households", "redemptions"]].sort_values("start_date"))

## 1. Which campaigns can we compare?

The guide focuses on large Type A campaigns 8, 13 and 18. Campaign 27 begins too early for the required eight-week pre-period. Campaign assignment used past purchases, so targeted households started out different from everyone else.

In [ ]:
roi = load("campaign_roi_inputs")
display(roi[["campaign_id", "start_week", "end_week", "weeks", "households", "redemptions", "redeeming_households"]])
print("Grocery households x 52 weeks:", 2_467 * 52)

## 2. Follow the estimate as the design improves

The naive gap compares spending during the campaign. Difference-in-differences (DiD) subtracts each group's earlier spending. Matching also balances the groups on prior spending and activity. The earlier-window method matches on weeks separate from the eight-week before-period to reduce regression-to-the-mean bias.

In [ ]:
est = load("guide_compatibility_estimates")
est = est[est.campaign_id.astype(str).isin(["8", "13", "18"])].copy()
est["interval"] = est.apply(lambda r: f"({r.ci_low:+.2f}, {r.ci_high:+.2f})", axis=1)
display(est.pivot(index="campaign_id", columns="method", values="estimate").round(2))
display(est[["campaign_id", "method", "n_treated", "n_control", "estimate", "ci_low", "ci_high"]].round(2))

The naive +$32–$35 weekly gap is not incremental lift: it largely reflects who was selected. Pre-period matching creates a larger estimate, but a campaign fake-start can create a similar apparent lift before the real campaign. Earlier-window matching yields smaller estimates whose intervals include zero.

In [ ]:
placebo = load("placebo_tests")
placebo = placebo[placebo.campaign_id.astype(str).isin(["8", "13", "18"])]
display(placebo[["campaign_id", "method", "status", "estimate", "ci_low", "ci_high", "p_holm"]].round(3))

Campaign 8's earlier-window placebo is not estimable because its fake-start would require weeks before the dataset begins. Campaigns 13 and 18 have earlier-window placebo intervals that include zero. A passing placebo does not prove the real estimate is causal; it only removes one warning.

In [ ]:
display(load("campaign_overlap_audit"))
display(load("campaign_sensitivities")[["campaign_id", "method", "n_treated", "n_control", "estimate", "ci_low", "ci_high"]].round(2))
display(load("pretrend_tests")[["campaign_id", "contrast", "estimate", "ci_low", "ci_high", "p_holm"]].round(3))
display(load("campaign_window_audit"))

## 3. Inspect the weekly pattern and the money inputs

The weekly chart shows the noisy gap for Campaign 18. Pointwise intervals are separate from the pre-period contrasts above and do not validate parallel trends. Campaigns overlap, and the source has no print or distribution cost data. Whole source weeks approximate campaign dates: Campaign 13 includes one day before its actual Tuesday start.

In [ ]:
display(Image(filename=str(ROOT / "charts" / "campaign18_pretrend.png")))
display(load("campaign_roi_inputs"))
display(load("coupon_amount_summary"))

## 4. Run the workflow yourself in DBeaver

Open `grocery_campaigns.duckdb` with `Open Campaign Analysis in DBeaver.cmd`, then run one statement at a time from `sql/05_campaigns_learning.sql` with Ctrl+Enter. These are small reporting tables. The full household-week model and statistical estimates are reproducible with `scripts/run_campaigns.py`.

**Conclusion:** the best observational design finds a small, uncertain spend change. Recommend a randomized household holdout and measure net revenue and retailer costs before claiming campaign ROI.

In [ ]:
checks = load("campaign_checks")
assert checks["passed"].astype(str).str.lower().eq("true").all()
print(f"{len(checks)} campaign checks passed")
display(checks)